In [1]:
# =========================================
# Imports & config
# =========================================

import pandas as pd
import numpy as np
import pyodbc
import folium
from branca.colormap import LinearColormap
from sklearn.neighbors import BallTree

GRID_SIZE_DEG = 0.01
MAX_MATCH_DEG = 0.005

colormap = LinearColormap(
    colors=["green", "yellow", "orange", "red"],
    vmin=1,
    vmax=10
)

In [3]:
# =========================================
# Connections
# =========================================

conn_moody = """
DRIVER={ODBC Driver 17 for SQL Server};
SERVER=wagas;
DATABASE=KMIS_Temp;
Trusted_Connection=yes;
"""

conn_policy = """
DRIVER={ODBC Driver 17 for SQL Server};
SERVER=prodsnapshot;
DATABASE=KMIS_HOPS_SS;
Trusted_Connection=yes;
"""

def read_sql(sql, conn_str):
    with pyodbc.connect(conn_str) as conn:
        return pd.read_sql(sql, conn)

In [4]:
# =========================================
# SQL
# =========================================

SQL_MOODY = """
SELECT
    LocationRiskId,
    Latitude,
    Longitude,
    Score100yr,
    Score250yr,
    Score500yr
FROM KMIS_Temp.dbo.Moodys_Location_Risk
"""

SQL_POLICY = """
SELECT
[TN].TextTrackingNumber AS [POLICY NUMBER],
[HH].LATITUDE AS [LATITUDE],
[HH].LONGITUDE AS [LONGITUDE],
[AA].Address1 AS [STREET],
[AA].City AS [CITY],
[SA].StateName AS [STATE],
[AA].Zipcode AS [ZIP CODE], 
[ST].SubmissionTypeName AS [PRODUCT],
CASE WHEN [PL].AttachmentPoint IS NULL THEN 0 
     ELSE [PL].AttachmentPoint 
END AS [ATTACHMENT],
CASE WHEN FrontingCarrier = 'Knight Specialty Insurance Services' 
     THEN [PINV].InvoiceAmount
     ELSE KnightPremium 
END AS [PREMIUM],
CASE 
    WHEN [ST].SubmissionTypeName = 'Primary' 
         AND FrontingCarrier ='Knight Specialty Insurance Services' 
    THEN ISNULL([PRC].LimitCoverageA, 0) 
       + ISNULL([PRC].LimitCoverageB, 0) 
       + ISNULL([PRC].LimitCoverageC, 0) 
       + ISNULL([PRC].LimitCoverageD, 0)
    WHEN [ST].SubmissionTypeName = 'Excess' 
         AND [FC].FrontingCarrier<>'Knight Specialty Insurance Services'
         AND ReinsurerName = 'Knight Specialty Insurance Company' 
    THEN [PL].LayerLimit * [QS].QuotaShare / 100
    WHEN UPPER([ST].SubmissionTypeName) = 'EXCESS' 
         AND FrontingCarrier ='Knight Specialty Insurance Services'
    THEN [PL].LayerLimit 
END AS [LIMIT],
CASE 
    WHEN [ST].SubmissionTypeName = 'Primary' 
    THEN [PRC].LimitCoverageA + [PRC].LimitCoverageB
    WHEN [ST].SubmissionTypeName = 'Excess' 
    THEN [PRC].StatedCoverageA 
END AS [BUILDING VALUE],
[PRC].LimitCoverageC AS [CONTENTS VALUE],
[PRC].LimitCoverageD AS [BUSINESS INTERRUPTION VALUE],
CONCAT(TRIM([HH].WildfireScore), '-', TRIM([HH].WildfireBase)) AS [HH SCORE]

FROM [KMIS_HOPS_SS].[Policy].[Policy] [P]
INNER JOIN [KMIS_HOPS_SS].[Policy].[Root] [R] 
    ON [R].LatestPolicyID = [P].[PolicyID]
LEFT JOIN [KMIS_HOPS_SS].Policy.Property [PR]
    ON [PR].PropertyID = [P].PropertyID
LEFT JOIN [KMIS_HOPS_SS].Policy.PropertyCoverage [PRC]
    ON [PRC].PropertyID = [PR].PropertyID
LEFT JOIN [KMIS_HOPS_SS].Address.Address [AA] 
    ON [AA].AddressID = [PR].AddressID
LEFT JOIN [KMIS_HOPS_SS].[Policy].[PolicyInsured] [PI] 
    ON [PI].PolicyID = [P].PolicyID
LEFT JOIN [KMIS_HOPS_SS].Customer.Insured [CI] 
    ON [CI].InsuredID = [PI].InsuredID
LEFT JOIN [KMIS_HOPS_SS].[Policy].[HazardHub] [HH] 
    ON [HH].HazardHubID = [P].HazardHubID
LEFT JOIN [KMIS_HOPS_SS].[Policy].[Submission_Layer] [SL] 
    ON [SL].PolicyID = [P].PolicyID
LEFT JOIN [KMIS_HOPS_SS].[Policy].[Layer] [PL] 
    ON [PL].LayerID = [SL].LayerID
LEFT JOIN [KMIS_HOPS_SS].[Policy].[Layer_Quotashare] [LQ]
    ON [LQ].LayerID = [SL].LayerID
LEFT JOIN [KMIS_HOPS_SS].[Policy].[Quotashare] [QS]
    ON [QS].QuotashareID = [LQ].QuotashareID
LEFT JOIN [KMIS_HOPS_SS].[Lookup].[FrontingCarrier] [FC] 
    ON [FC].FrontingCarrierID = [PL].FrontingCarrierID
LEFT JOIN [KMIS_HOPS_SS].[Broker].[BrokerTeam] [BT] 
    ON [BT].BrokerTeamID = [P].BrokerTeamID
LEFT JOIN [KMIS_HOPS_SS].[Broker].[BrokerOffice] [BO] 
    ON [BO].BrokerOfficeID = [BT].BrokerofficeID
LEFT JOIN [KMIS_HOPS_SS].[Policy].[TrackingNumber] [TN] 
    ON [TN].TrackingNumberID = [P].TrackingNumberID
LEFT JOIN [KMIS_HOPS_SS].[Policy].[Invoices] [PINV]
    ON [PINV].InvoiceID = [P].InvoiceID
LEFT JOIN [KMIS_HOPS_SS].[Lookup].[State] [SA] 
    ON [SA].StateID = [AA].StateID
LEFT JOIN [KMIS_HOPS_SS].[Lookup].[SubmissionType] [ST] 
    ON [ST].SubmissionTypeID = [P].SubmissionTypeID
LEFT JOIN [KMIS_HOPS_SS].[Lookup].[Status] [LS]
    ON [LS].StatusID = [P].StatusID
LEFT JOIN [KMIS_HOPS_SS].[Lookup].[Reinsurer] [RS]
    ON [RS].ReinsurerID = [QS].ReinsurerID

WHERE ([FC].FrontingCarrier='Knight Specialty Insurance Services' 
    OR [RS].ReinsurerName='Knight Specialty Insurance Company')
AND PrimaryContact = 1
AND CI.NamedInsured NOT LIKE '%Test%'
AND BO.Name NOT LIKE '%Test%'
AND StatusName IN ('Active', 'Bound')
"""

In [5]:
# =========================================
# Load data
# =========================================

moody_df = read_sql(SQL_MOODY, conn_moody)
policy_df = read_sql(SQL_POLICY, conn_policy)

moody_df.columns = moody_df.columns.str.upper()
policy_df.columns = policy_df.columns.str.upper()

print(moody_df.shape, policy_df.shape)

C:\Users\XDong\AppData\Local\Temp\ipykernel_38932\3275317039.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


(326, 6) (213, 15)


In [6]:
# =========================================
# Matching
# =========================================

policy_df["LAT_R"] = policy_df["LATITUDE"].round(5)
policy_df["LON_R"] = policy_df["LONGITUDE"].round(5)

moody_df["LAT_R"] = moody_df["LATITUDE"].round(5)
moody_df["LON_R"] = moody_df["LONGITUDE"].round(5)

moody_subset = moody_df[[
    "LAT_R", "LON_R",
    "SCORE100YR", "SCORE250YR", "SCORE500YR"
]]

policy_merged = policy_df.merge(
    moody_subset,
    on=["LAT_R", "LON_R"],
    how="left"
)

missing = policy_merged["SCORE100YR"].isna()

if missing.sum() > 0:
    tree = BallTree(
        np.deg2rad(moody_df[["LATITUDE", "LONGITUDE"]]),
        metric="haversine"
    )

    coords = np.deg2rad(policy_merged.loc[missing, ["LATITUDE", "LONGITUDE"]])
    dist, idx = tree.query(coords, k=1)

    nearest = moody_df.iloc[idx.flatten()].reset_index(drop=True)

    for i, index in enumerate(policy_merged.loc[missing].index):
        if np.rad2deg(dist[i][0]) <= MAX_MATCH_DEG:
            policy_merged.loc[index, "SCORE100YR"] = nearest.loc[i, "SCORE100YR"]
            policy_merged.loc[index, "SCORE250YR"] = nearest.loc[i, "SCORE250YR"]
            policy_merged.loc[index, "SCORE500YR"] = nearest.loc[i, "SCORE500YR"]

print(policy_merged["SCORE100YR"].isna().sum())

0


In [7]:
# =========================================
# Grid assignment
# =========================================

lat_min = policy_merged["LATITUDE"].min() - GRID_SIZE_DEG
lon_min = policy_merged["LONGITUDE"].min() - GRID_SIZE_DEG

def get_cell(lat, lon):
    r = int((lat - lat_min) // GRID_SIZE_DEG)
    c = int((lon - lon_min) // GRID_SIZE_DEG)
    return r, c

policy_merged["CELL_ROW"], policy_merged["CELL_COL"] = zip(
    *policy_merged.apply(lambda x: get_cell(x["LATITUDE"], x["LONGITUDE"]), axis=1)
)

policy_merged["CELL_ID"] = (
    policy_merged["CELL_ROW"].astype(str) + "_" +
    policy_merged["CELL_COL"].astype(str)
)

In [8]:
# =========================================
# Aggregation
# =========================================

damage_curve = {
    1: 0.005, 2: 0.01, 3: 0.05, 4: 0.10, 5: 0.15,
    6: 0.20, 7: 0.30, 8: 0.40, 9: 0.50, 10: 1.00
}

def loss_rate(score):
    if pd.isna(score):
        return 0
    s = int(round(score))
    s = max(1, min(10, s))
    return damage_curve[s]

def weighted_score(df, col):
    w = df["BUILDING VALUE"].sum()
    if w == 0:
        return np.nan
    return (df[col] * df["BUILDING VALUE"]).sum() / w

rows = []

for cid, g in policy_merged.groupby("CELL_ID"):
    bv = g["BUILDING VALUE"].sum()
    limit = g["LIMIT"].sum()
    prem = g["PREMIUM"].sum()

    s100 = weighted_score(g, "SCORE100YR")
    s250 = weighted_score(g, "SCORE250YR")
    s500 = weighted_score(g, "SCORE500YR")

    lr100 = (limit * loss_rate(s100) / prem)
    lr250 = (limit * loss_rate(s250) / prem)
    lr500 = (limit * loss_rate(s500) / prem)

    rows.append({
        "CELL_ROW": g["CELL_ROW"].iloc[0],
        "CELL_COL": g["CELL_COL"].iloc[0],

        "S100": s100,
        "S250": s250,
        "S500": s500,

        "LIMIT": limit,
        "PREMIUM": prem,

        "PML100": limit * loss_rate(s100),
        "PML250": limit * loss_rate(s250),
        "PML500": limit * loss_rate(s500),

        "LR100": lr100,
        "LR250": lr250,
        "LR500": lr500
    })

grid_df = pd.DataFrame(rows)
print(len(grid_df))

144


In [10]:
# =========================================
# Map
# =========================================

from branca.element import Template, MacroElement

center = [
    policy_merged["LATITUDE"].mean(),
    policy_merged["LONGITUDE"].mean()
]

m = folium.Map(location=center, zoom_start=9, tiles=None)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri"
).add_to(m)

def bounds(r, c):
    lat1 = lat_min + r * GRID_SIZE_DEG
    lat2 = lat1 + GRID_SIZE_DEG
    lon1 = lon_min + c * GRID_SIZE_DEG
    lon2 = lon1 + GRID_SIZE_DEG
    return [[lat1, lon1], [lat2, lon2]]

# ---------- GRID ----------
for _, r in grid_df.iterrows():

    b = bounds(r["CELL_ROW"], r["CELL_COL"])

    # color function
    def lr_color(lr):
        if pd.isna(lr):
            return "black"
        elif lr < 0.7:
            return "green"
        elif lr < 1:
            return "orange"
        else:
            return "red"

    c100 = lr_color(r["LR100"])
    c250 = lr_color(r["LR250"])
    c500 = lr_color(r["LR500"])

    popup = f"""
    <div style="width:240px; font-family:Arial; font-size:12px; line-height:1.3;">

    <b>Risk Score (100yr):</b> {r['S100']:.2f}<br>
    <b>Risk Score (250yr):</b> {r['S250']:.2f}<br>
    <b>Risk Score (500yr):</b> {r['S500']:.2f}<br>

    <br>

    <b>Limit:</b> ${r['LIMIT']:,.0f}<br>
    <b>Premium:</b> ${r['PREMIUM']:,.0f}<br>

    <br>

    <b>Estimated Loss (100yr):</b> ${r['PML100']:,.0f}<br>
    <b>LR (100yr):</b> <span style="color:{c100}; font-weight:600;">{r['LR100']:.1%}</span><br>

    <b>Estimated Loss (250yr):</b> ${r['PML250']:,.0f}<br>
    <b>LR (250yr):</b> <span style="color:{c250}; font-weight:600;">{r['LR250']:.1%}</span><br>

    <b>Estimated Loss (500yr):</b> ${r['PML500']:,.0f}<br>
    <b>LR (500yr):</b> <span style="color:{c500}; font-weight:600;">{r['LR500']:.1%}</span><br>

    </div>
    """

    folium.Rectangle(
        bounds=b,
        fill=True,
        fill_color=colormap(r["S100"]),
        fill_opacity=0.6,
        color=None,
        popup=folium.Popup(popup, max_width=280),
        class_name="layer100"
    ).add_to(m)

    folium.Rectangle(
        bounds=b,
        fill=True,
        fill_color=colormap(r["S250"]),
        fill_opacity=0.6,
        color=None,
        popup=folium.Popup(popup, max_width=280),
        class_name="layer250"
    ).add_to(m)

    folium.Rectangle(
        bounds=b,
        fill=True,
        fill_color=colormap(r["S500"]),
        fill_opacity=0.6,
        color=None,
        popup=folium.Popup(popup, max_width=280),
        class_name="layer500"
    ).add_to(m)

# ---------- POLICY ----------
for _, r in policy_merged.iterrows():

    if r["PRODUCT"].upper() == "EXCESS":
        attachment_line = f"<b>Attachment:</b> ${r['ATTACHMENT']:,.0f}<br>"
    else:
        attachment_line = ""

    popup = f"""
    <div style="width:240px; font-family:Arial; font-size:12px; line-height:1.3;">

    <b>Policy:</b> {r['POLICY NUMBER']}<br>
    <b>Address:</b> <span style="white-space:nowrap;">{r['STREET']}, {r['CITY']}</span><br>
    <b>Product:</b> {r['PRODUCT']}<br>
    {attachment_line}
    <b>Limit:</b> ${r['LIMIT']:,.0f}<br>
    <b>Premium:</b> ${r['PREMIUM']:,.0f}<br>
    <b>HH Score:</b> {r['HH SCORE']}

    </div>
    """

    folium.CircleMarker(
        location=[r["LATITUDE"], r["LONGITUDE"]],
        radius=3,
        color="black",
        fill=True,
        fill_opacity=1,
        popup=folium.Popup(popup, max_width=280)
    ).add_to(m)

# ---------- LEGEND ----------
colormap.caption = "Weighted RMS Risk Score"
colormap.add_to(m)

# ---------- TOGGLE UI ----------
template = """
{% macro html(this, kwargs) %}

<div id="toggle-container" style="
position: fixed;
top: 10px;
left: 50px;
z-index:9999;
background: white;
padding: 8px;
border-radius: 6px;
box-shadow: 0 0 6px rgba(0,0,0,0.2);
font-size:12px;
">

<button id="btn100" class="toggle-btn active" onclick="showLayer('layer100', 'btn100')">100yr</button>
<button id="btn250" class="toggle-btn" onclick="showLayer('layer250', 'btn250')">250yr</button>
<button id="btn500" class="toggle-btn" onclick="showLayer('layer500', 'btn500')">500yr</button>

</div>

<style>
.toggle-btn {
    margin-right:4px;
    padding:4px 8px;
    border:none;
    border-radius:4px;
    background:#eee;
    cursor:pointer;
}

.toggle-btn:hover {
    background:#ddd;
}

.toggle-btn.active {
    background:#333;
    color:white;
}
</style>

<script>

function showLayer(layer, btnId) {

    document.querySelectorAll('.layer100, .layer250, .layer500')
        .forEach(el => el.style.display = 'none');

    document.querySelectorAll('.' + layer)
        .forEach(el => el.style.display = 'block');

    document.querySelectorAll('.toggle-btn')
        .forEach(btn => btn.classList.remove('active'));

    document.getElementById(btnId).classList.add('active');
}

window.onload = function() {
    showLayer('layer100', 'btn100');
}

</script>

{% endmacro %}
"""

macro = MacroElement()
macro._template = Template(template)
m.get_root().add_child(macro)

# ---------- SAVE ----------
m.save("map.html")